# Week 6

### Goals 
- Combine preference extraction, tool choice and catalog-grounded recommendations.
- Produce reasons that cite available movie attributes.

### Tasks 
- Pass the user request and available tool definitions to the agent.
- Let the agent select search, filter and ranking actions within a step limit.
- Return one primary recommendation and one alternative from catalog IDs.
- Explain the match using fields that actually exist in the catalog.
- Handle empty results by relaxing a stated preference or asking for a revised request.
- Compare the agent output with the Week 3 baseline on a few requests.

### Expected Output
- 06_recommendation_agent.ipynb with an end-to-end run.
- Visible action trace and a brief comparison with the baseline.

---
For these taks I follow the structure set up in the tasks of week 5, but with the necessary adjustments and improvements to fullfill the week 6 tasks.  


#### Imports and tools

In [4]:
# Base imports
import sys
from ollama import chat

# Custom tools for agent
sys.path.append('../')
from src.tools import search_movies, filter_movies, rank_candidates, get_movie_details

# Expose the tools to the agent
ALL_TOOLS = [search_movies, filter_movies, rank_candidates, get_movie_details]
tool_map = {tool.__name__: tool for tool in ALL_TOOLS}

#### System Prompt

In [5]:
SYSTEM_INSTRUCTION = """You are a strict, data-driven movie recommendation agent. You have access to a local movie catalog via tools.

CRITICAL RULES:
1. NO EXTERNAL MEMORY: Never invent, guess, or hallucinate movies, plots, runtimes, actors, or ratings. Only use facts explicitly returned by your tools.
2. ZERO-MATCH FALLBACK: If a tool returns an empty list ([]), you must search again with broader parameters (e.g., omit the year) using the proper tool call. DO NOT type raw JSON in your text response. 
3. TWO RECOMMENDATIONS: When you find matching movies, you MUST provide EXACTLY two movies: one Primary and one Alternative.
4. OBSERVED FIELDS ONLY: Explain why you recommend each movie using ONLY the exact fields provided in the tool observation (e.g., 'genres', 'year', 'mean').

RESPONSE FORMAT:
Primary Recommendation: [Movie Title] (ID: [movieId])
- Why it matches: [Cite specific fields like year, genre, or mean rating from observation]

Alternative: [Movie Title] (ID: [movieId])
- Why it matches: [Cite specific fields]
"""

#### Agent execution loop

In [8]:
def run_recommendation_agent(user_prompt, max_steps=5):
    messages = [
        {'role': 'system', 'content': SYSTEM_INSTRUCTION},
        {'role': 'user', 'content': user_prompt}
    ]
    
    step_count = 0
    print(f"\n=== AGENT TRACE FOR: '{user_prompt}' ===")
    
    while step_count < max_steps:
        step_count += 1
        print(f"\n--- Step {step_count} ---")
        
        response = chat(model='llama3.1:8b', messages=messages, tools=ALL_TOOLS)
        messages.append(response.message)
        
        # DECIDE PHASE
        if response.message.tool_calls:
            for tc in response.message.tool_calls:
                func_name = tc.function.name
                args = tc.function.arguments
                print(f"Action (Decide): The LLM wants to call {func_name}({args})")
                
                # ACT & OBSERVE PHASE
                if func_name in tool_map:
                    try:
                        # Execute the python function
                        result = tool_map[func_name](**args)
                        
                        # Truncate output for readable notebook prints
                        obs_str = str(result)
                        print_str = obs_str[:200] + "..." if len(obs_str) > 200 else obs_str
                        print(f"Observation (Act): {print_str}")
                        
                        # Feed the factual observation back to the LLM
                        messages.append({'role': 'tool', 'tool_name': func_name, 'content': obs_str})
                    
                    except Exception as e:
                        # Defensive programming: Catch type mismatches and tell the LLM it messed up
                        error_msg = f"Tool Error: {str(e)}. Check your argument types (e.g., ensure year is an integer, not a string)."
                        print(f"Observation (Error): {error_msg}")
                        messages.append({'role': 'tool', 'tool_name': func_name, 'content': error_msg})
        else:
            # RESPOND PHASE
            print("\n=== FINAL ANSWER ===")
            print(response.message.content)
            break

    if step_count >= max_steps:
        print("\n[System] Maximum steps reached. Agent stopped to prevent infinite loop.")

#### Test simple end-to-end runs

In [9]:
# 1. Standard Request (Tests primary/alternative formatting and field citations)
run_recommendation_agent("Find me a highly rated Sci-Fi movie from 1999.")

# 2. Zero-Match Fallback (Tests how it handles an empty list, no action movie from 1905 exists in the dataset)
run_recommendation_agent("I want an Action movie from 1905.")


=== AGENT TRACE FOR: 'Find me a highly rated Sci-Fi movie from 1999.' ===

--- Step 1 ---
Action (Decide): The LLM wants to call filter_movies({'genre': 'Sci-Fi', 'year': '1999'})
Observation (Act): [{'movieId': 2448, 'title': 'Virus', 'genres': ['Horror', 'Sci-Fi'], 'mean': 2.6666666666666665, 'count': 3, 'year': 1999.0}, {'movieId': 2498, 'title': 'My Favorite Martian', 'genres': ['Comedy', 'Sc...

--- Step 2 ---

=== FINAL ANSWER ===
Primary Recommendation: The Matrix (ID: 2571)
- Why it matches: High mean rating (4.192446043165468) and large number of ratings (278)

Alternative: Galaxy Quest (ID: 3175)
- Why it matches: High mean rating (3.58) and relatively large number of ratings (75) in the Sci-Fi genre.

=== AGENT TRACE FOR: 'I want an Action movie from 1905.' ===

--- Step 1 ---
Action (Decide): The LLM wants to call filter_movies({'genre': 'Action', 'year': 1905})
Observation (Act): []

--- Step 2 ---

=== FINAL ANSWER ===
Since the tool call returned an empty list, I will b

#### Weel 3 baseline comparison

In [10]:
# Baseline function from week 3
# This was only a simple rule-based recommendation function, without any LLM or agent logic
def recommend_movies(catalog_df, target_genre=None, target_year=None, min_ratings=3, top_n=5):
    # Create a copy so the original dataframe is not modified
    candidates = catalog_df.copy()
    
    # Account for few ratings by applying the minimum threshold
    candidates = candidates[candidates['count'] >= min_ratings]
    
    # Filter by year
    if target_year is not None:
        candidates = candidates[candidates['year'] == target_year]
        
    # Filter by genre (checking if the target_genre is in the list of genres)
    if target_genre is not None:
        candidates = candidates[candidates['genres'].apply(lambda x: target_genre in x)]

    # If no candidates remain after filtering, return an empty list
    if candidates.empty:
        return []
        
    # Rank candidates by simple transparent score (mean rating)
    candidates = candidates.sort_values(by='mean', ascending=False)
    
    # Take the top N rows BEFORE converting to a list of dictionaries
    top_candidates = candidates.head(top_n).to_dict('records')
    
    # Keep unknown fields marked as unknown
    for row in top_candidates:
        row['runtime'] = 'Unknown'
        row['plot'] = 'Unknown'

    return top_candidates

In [11]:
# Load the catalog
import pandas as pd
catalog_df = pd.read_csv('../data/cleaned_movie_data.csv')

In [12]:
recommend_movies(catalog_df, target_genre='Sci-Fi', target_year=1999)

[{'movieId': 2571,
  'title': 'The Matrix',
  'genres': "['Action', 'Sci-Fi', 'Thriller']",
  'mean': 4.192446043165468,
  'count': 278,
  'year': 1999.0,
  'runtime': 'Unknown',
  'plot': 'Unknown'},
 {'movieId': 2761,
  'title': 'The Iron Giant',
  'genres': "['Adventure', 'Animation', 'Children', 'Drama', 'Sci-Fi']",
  'mean': 4.051282051282051,
  'count': 39,
  'year': 1999.0,
  'runtime': 'Unknown',
  'plot': 'Unknown'},
 {'movieId': 2600,
  'title': 'eXistenZ',
  'genres': "['Action', 'Sci-Fi', 'Thriller']",
  'mean': 3.863636363636364,
  'count': 22,
  'year': 1999.0,
  'runtime': 'Unknown',
  'plot': 'Unknown'},
 {'movieId': 3175,
  'title': 'Galaxy Quest',
  'genres': "['Adventure', 'Comedy', 'Sci-Fi']",
  'mean': 3.58,
  'count': 75,
  'year': 1999.0,
  'runtime': 'Unknown',
  'plot': 'Unknown'},
 {'movieId': 7810,
  'title': 'Babylon 5: A Call to Arms',
  'genres': "['Adventure', 'Sci-Fi']",
  'mean': 3.333333333333333,
  'count': 3,
  'year': 1999.0,
  'runtime': 'Unknown',

In [13]:
recommend_movies(catalog_df, target_genre='Action', target_year=1905)

[]

### Observations
The baseline from week 3 is simply a basic, deterministic Python function, with no LLM or agent logic. Therefore, users cannot interact with it using full-text input, and the output is just a list of dictionaries rather than a readable text containing sentences. The input parameters of the function have to be provided correctly, which is, of course, not user-friendly for people without programming knowledge. However, this baseline version has one significant advantage over the agent version: speed. It takes less than a second to run.   

The agentic workflow is much easier to understand in terms of input and output readability, but there are some significant drawbacks. With the llama3.1:8b model, processing one request takes roughly 40 seconds. The model must understand the request, select the appropriate tools, execute these tools, analyse the results and then generate a final response. Similar to the tasks of last week (5), issues with small LLM models are evident throughout this entire process: They understand the logic of multi-step tools, but struggle with execution. They frequently suffer from 'tool-call bleeding', as can be seen in the second trace. Instead of properly triggering the actual tool call, the model types out a raw JSON format in its final text response. In the first trace, the model also bypassed the call to the `rank_candidates` tool. Instead, it simply read the mean ratings directly from the output of the `filter_movies` tool and chose the ranking itself.   

I tried multiple solutions to improve the model's performance. I adjusted the system prompt and tested various models. I started with Llama 3.2, which was used before, and then tested newer, more sophisticated and larger local models, such as Qwen 3.5:9b. I also adjusted the function in the `src/tools.py` file, but nothing seemed to solve the issue fully. For example, the Qwen3.5 model with 9 billion parameters, which should theoretically be a strong and efficient local model, took approximately 22 minutes just to run the two above traces. It was the only model that ran multiple steps instead of providing the final answer in step 2, but it still malfunctioned as it repeated some tool calls unnecessarily and made simple mistakes that the smaller Llama 3.2 model did not make. It seemed to overcomplicate things by not fully following the instructions and trying to do things its own way, resulting in failure. Based on my current knowledge, I suspect that there are errors in my workflow regarding how the agent interacts with the tools, or that the smaller local models are simply incapable of correctly following all instructions without losing track of the steps and either hallucinating or simply stopping.  

*Note on how empty results currently are handeled:*   
In this version there is just a fallback for the agent to automatically use a tool call again but with relaxed parametrs (for example if for genre 'action' and year '1905' nothing is found), it may only use one parameter (in this case the genre) as sole input instead of also filtering for all arguments. This means the agent decides that by itself, without consulting the user. For the final version a combination may be better: Ask the user to on which parameter(s) the focus should be on, instead of letting the agent decide for itself. The current difficulty with this better approach is the size of the model: as discussed previously, the currently used small models start to break down already after the fist or second step and larger models take extremly long and/or start going their own way instead of directly following instructions.
